In [1]:
import pandas as pd

In [18]:
from collections import defaultdict

In [59]:
import numpy as np

In [34]:
from itertools import combinations

In [55]:
cre=pd.read_csv("Cafe_priap_GO_rr.tsv",sep='\t')

In [12]:
#crer=cre[(cre['Enrich']=='e') & (cre['p_fdr_bh']<0.01)]

In [61]:
cre

,Ontology,Term,Name,Enrich,Study,Pop,p_fdr_bh,p_uncor,Node,Nod,Proc,GeneFamInStudy
0,BP,GO:0050817,coagulation,e,10,286,0.011969,0.000002,Cryptovermes_cont,Cryptovermes,cont,"Narfl|Rl21|Unchar_15503,Nav3-2,Unchar_14757,Un..."
1,BP,GO:0007599,hemostasis,e,10,294,0.011969,0.000002,Cryptovermes_cont,Cryptovermes,cont,"Narfl|Rl21|Unchar_15503,Nav3-2,Unchar_14757,Un..."
2,BP,GO:0002455,humoral immune response mediated by circulatin...,e,4,20,0.011969,0.000003,Cryptovermes_cont,Cryptovermes,cont,"Narfl|Chac2|Rs15a|Nptx2-20,Nptx1-2,Int9-1,Nptx..."
3,BP,GO:0007596,blood coagulation,e,9,275,0.026501,0.000011,Cryptovermes_cont,Cryptovermes,cont,"Narfl|Rl21|Unchar_15503,Nav3-2,Unchar_14757,Un..."
4,BP,GO:0034220,monoatomic ion transmembrane transport,e,16,927,0.028995,0.000013,Cryptovermes_cont,Cryptovermes,cont,"Ach3-5,Ach3-1,Ach3-3,Ach3-2,Ach3-6,Ach3-4,Unch..."
...,...,...,...,...,...,...,...,...,...,...,...,...
2290,BP,GO:0055085,transmembrane transport,e,7,1232,0.026154,0.000006,Pancrustacea_exp,Pancrustacea,exp,||Unchar_2199|Gba2||Unchar_15850|
2291,BP,GO:0006811,monoatomic ion transport,e,7,1249,0.026154,0.000007,Pancrustacea_exp,Pancrustacea,exp,||Unchar_2199|Gba2||Unchar_15850|
2292,BP,GO:0008209,androgen metabolic process,e,3,49,0.026154,0.000007,Pancrustacea_exp,Pancrustacea,exp,|Unchar_2199|
2293,MF,GO:0015295,solute:proton symporter activity,e,3,40,0.026154,0.000004,Pancrustacea_exp,Pancrustacea,exp,|Unchar_15850|


In [56]:
git={}
for i,r in cre.iterrows():
    #print(r.Term,[g for gc in r.GeneFamInStudy.split('|') for g in gc.split(',') if not g==''])
    git[r.Term]=set([g for gc in str(r.GeneFamInStudy).split('|') for g in gc.split(',') if not g==''])
    

In [62]:
def jaccard_similarity(set1, set2):
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0

In [75]:
got=list(git.keys())

In [67]:
simMat = np.zeros((len(git.keys()),len(git.keys())))
for i,t1 in enumerate(list(git.keys())):
    for j,t2 in enumerate(list(git.keys())):
        simMat[i, j] = jaccard_similarity(git[t1],git[t2])    

In [68]:
simMat

array([[1.        , 1.        , 0.16666667, ..., 0.        , 0.        ,
        0.        ],
       [1.        , 1.        , 0.16666667, ..., 0.        , 0.        ,
        0.        ],
       [0.16666667, 0.16666667, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(2227, 2227))

In [71]:
from sklearn.cluster import AffinityPropagation

In [80]:
ap = AffinityPropagation(
    affinity='precomputed',  # Important: tells it you're providing similarities
    damping=0.7,             # Damping factor (0.5-1.0), higher = more conservative
    max_iter=200,            # Maximum iterations
    convergence_iter=15,     # Iterations without change to declare convergence
    random_state=42
)

In [81]:
clusters = ap.fit_predict(simMat)


/Users/ferdi/miniforge3/envs/pyb/lib/python3.13/site-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagation did not converge, this model may return degenerate cluster centers and labels.
  warnings.warn(


In [79]:
 np.median(simMat)

np.float64(0.07142857142857142)

In [82]:
# Get cluster exemplars (representative GO terms)
exemplar_indices = ap.cluster_centers_indices_
n_clusters = len(exemplar_indices)
ex_go=[got[i] for i in exemplar_indices]
print(f"Number of clusters found: {n_clusters}")
print(f"Exemplar GO terms: {ex_go}")

Number of clusters found: 401
Exemplar GO terms: ['GO:0055085', 'GO:0008146', 'GO:0050698', 'GO:0090502', 'GO:0035590', 'GO:0002566', 'GO:0007220', 'GO:0009617', 'GO:0042632', 'GO:0009581', 'GO:0031093', 'GO:0043267', 'GO:1903317', 'GO:0001819', 'GO:0006955', 'GO:0043549', 'GO:0060603', 'GO:0010596', 'GO:1901889', 'GO:1901222', 'GO:2000026', 'GO:0030155', 'GO:1901343', 'GO:0050921', 'GO:0015833', 'GO:1903532', 'GO:0051384', 'GO:0051336', 'GO:0048523', 'GO:0009605', 'GO:0022414', 'GO:0051346', 'GO:0007204', 'GO:0048859', 'GO:0055082', 'GO:0048583', 'GO:0140352', 'GO:0048667', 'GO:2001258', 'GO:0003170', 'GO:0048699', 'GO:0016045', 'GO:0032677', 'GO:1900046', 'GO:0071384', 'GO:0007597', 'GO:0048232', 'GO:0110053', 'GO:0051493', 'GO:1903556', 'GO:0071214', 'GO:0043068', 'GO:0002283', 'GO:0050922', 'GO:0048477', 'GO:0002209', 'GO:0071671', 'GO:0071672', 'GO:1905521', 'GO:1904062', 'GO:0006807', 'GO:0070482', 'GO:0021955', 'GO:1903131', 'GO:1900274', 'GO:1904643', 'GO:0031344', 'GO:2000352'

In [83]:
with open('sel_terms.txt','x') as out: 
    out.write('\n'.join(ex_go))

In [43]:
import seaborn as sns

In [ ]:
#penguins = sns.load_dataset("penguins")
sns.histplot(simMat,binwidth=0.05)

In [57]:
print(len(git))

2227


In [60]:
simMat = np.zeros((2227, 2227))